In [6]:
import pandas as pd
import numpy as np

from tqdm.auto import tqdm

In [2]:
def calculate_statistics(reference, trim, tick_path, step_list=None):
    dic = {}
    for p in tqmd(tick_path):
        data = pd.read_csv(p)
        wafer_list = [wafer for wafer in data.columns if 'Wafer' in wafer]
        sensor_name = p.split(',')[-1].split('.csv')[0]
        dic[sensor_name] = {}
        for step in tqdm(step_list, desc='2nd loop'):
            step_list = []
            step_data = data[data.Step == step]
            step_data = step_data.reset_index(drop=True).iloc[trim:]

            if len(step_data) < 300 or step_data[reference].mean() is np.nan:
                continue
            else:
                norm = step_data.copy()
                norm[wafer_list] = norm[wafer_list] - norm[reference].mean(skip=True)

                if sum(norm.std(numeric_only=True, axis=0)[wafer_list] < 0.001) >= round(len(wafer_list) *0.7):
                    std_value = 1
                else:
                    std_value = norm[reference].std(skipna=True) + 0.0001
                norm[wafer_list] = norm[wafer_list] / std_value

                # calcualte statistics
                mean = abs(norm[wafer_list].mean(axis=0))
                std = norm[wafer_list].std(axis=0)
                min = norm[wafer_list].min(axis=0)
                max = norm[wafer_list].max(axis=0)
                p2p = max - min

                stat_list = [mean, std, min, max, p2p]

                stat_df = pd.concat(stat_list, axis=1).reset_index(drop=True)
                df = pd.concat([pd.DataFrame(wafer_list), stat_df], axis=1)
                df.columns = ['wafer', 'mean', 'std', 'min', 'max', 'p2p']

                dic[sensor_name][step] = df

    return dic

In [3]:
def calculate_distance(df, reference):
    feature_col = [col for col in df.columns if col != 'Wafer']
    ref_stat = df[df.wafer == reference][feature_col].values
    x_stat = df[feature_col].values

    distance = np.linalg.norm(x_stat - ref_stat, axis=1)
    #df['distance'] = distance
    #return df
    return distance

In [4]:
def calculate_score(df, reference, sigma=1):
    distance = calculate_distance(df, reference)
    k = np.exp(-distance / sigma)

    mean_value = np.mean(k)
    return 1 - mean_value

In [7]:
sensor_dic = {}
for sensor, dic in tqdm(precoat_dic.items()):
    sensor_dic[sensor] = {}
    for step, df in tqdm(dic.items()):
        sensor_dic[sensor][step] = calculate_score(df, reference)

NameError: name 'precoat_dic' is not defined